# Setup

## Imports

In [1]:
from pyaedt import Hfss
import os
import pandas as pd
from scipy.constants import pi, c, micro, milli, kilo, giga
import numpy as np
import matplotlib.pyplot as plt
import win32com.client
import gc
from psutil import cpu_count
from pybbq import BBQAnalysis

In [2]:
plt.style.use("tex.mplstyle")
%matplotlib widget

## Required Directories

In [3]:
dirs = [
    "hfss_files/snail_coffin_cavity",
    "data/eigenmode",
    "data/driven_modal",
    "data/bbq",
    "figures",
]


for dir in dirs:
    if not os.path.exists(dir):
        os.makedirs(dir)
        print(f"Created '{dir}'")

Created 'hfss_files/snail_coffin_cavity'
Created 'data/eigenmode'
Created 'data/driven_modal'
Created 'data/bbq'
Created 'figures'


## Initialize HFSS

In [4]:
# initialize hfss
project = os.getcwd() + "/hfss_files/snail_coffin_cavity/snail_coffin_cavity.aedt"

hfss = Hfss(
    project=project,
    design="EM_Cavity_with_Snail",
    solution_type="Eigenmode",
    remove_lock=True,
)

PyAEDT INFO: Python version 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.24.1.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\epm114\AppData\Local\Temp\pyaedt_epm114_26ad21d5-e261-45fa-85e5-db97411a26cb.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: New AEDT session is starting on gRPC port 54683.
PyAEDT INFO: Connecting to AEDT gRPC session on port 54683.
PyAEDT INFO: AEDT installation Path C:\Program Files\AnsysEM\v232\Win64
PyAEDT INFO: Client application successfully started.
PyAEDT INFO: 2023.2 version started with process ID 20468.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Project snail_coffin_cavity has been created.
PyAEDT INFO: Added design 'EM_Cavity_with_Snail' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read


In [ ]:
# cryogenic copper for the cavity walls (project-wide material, used by the
# "cryo_copper" finite-conductivity boundary in every design below)
# 5.8e9 S/m = OFHC copper at 4 K with RRR ~100; ignores anomalous skin effect
cryo_cu_conductivity = 5.8e9  # S/m

if not hfss.materials.exists_material("cryo_copper"):
    hfss.materials.add_material("cryo_copper")
hfss.materials["cryo_copper"].conductivity = cryo_cu_conductivity

## Inventor Parameters

In [5]:
# Get inventor user parameters
inv = win32com.client.Dispatch("Inventor.Application")

# Open the part file
doc = inv.Documents.Open(
    os.getcwd() + "\\inventor_files\\snail_coffin_cavity_bottom.ipt"
)

# Access the parameters collection
params = doc.ComponentDefinition.Parameters.UserParameters

# save to dict
parameters = dict()
for param in params:
    if param.Name == "theta":
        parameters[f"{param.Name}"] = param.Value * 180 / pi
    else:
        parameters[f"{param.Name}"] = param.Value * 10

# Close the document if it's not needed anymore
doc.Close()

# Quit the application if you started it and it's not needed anymore
inv.Quit()

# Release the COM objects
doc = None
params = None
inv = None

# Force garbage collection
gc.collect()

com_error: (-2147352567, 'Exception occurred.', (0, None, None, None, 0, -2147024809), None)

In [ ]:
parameters

In [ ]:
# define inventor params in hfss
for param, value in parameters.items():
    if param == "theta":
        hfss[f"${param}"] = f"{value}deg"
    else:
        hfss[f"${param}"] = f"{value}mm"

In [ ]:
sma_pin_length = 5
sma_pin_radius = 1.27 / 2

hfss["$sma_pin_length"] = f"{sma_pin_length}mm"
hfss["$sma_pin_radius"] = f"{sma_pin_radius}mm"

## Simulation Cores

In [ ]:
num_cores = cpu_count(logical=False)

# Eigenmode

## Cavity

In [ ]:
if "EM_Cavity" not in hfss.design_list:
    hfss.insert_design("EM_Cavity", "Eigenmode")
else:
    hfss.set_active_design("EM_Cavity")

# delete all objects
if hfss.modeler.get_objects_in_group("snail"):
    hfss.modeler.delete("snail")

if hfss.modeler.object_list:
    hfss.modeler.delete()

# import step file
hfss.modeler.import_3d_cad("inventor_files/snail_coffin_cavity.stp")

# create bounding box
box = hfss.modeler.get_group_bounding_box("snail_coffin_cavity")
box_min = box[:3]
box_max = box[3:]
box_origin = box_min
box_dim = [a - b for a, b in zip(box_max, box_min)]
cavity_params = {
    "origin": box_origin,
    "sizes": box_dim,
    "name": "cavity",
    "material": "vacuum",
}
cavity = hfss.modeler.create_box(**cavity_params)

# subtract
objs = hfss.modeler.object_names
hfss.modeler.subtract(objs[2], [objs[0], objs[1]], keep_originals=False)
hfss.modeler.ungroup("snail_coffin_cavity")

# sma pins
face_positions = [
    [0, "$cavity_y/2 + $sma_port_length - $sma_pin_length", "-$cavity_z/4"],
    [0, "$cavity_y/2 + $sma_port_length - $sma_pin_length", "$cavity_z/4"],
]
[
    hfss.modeler.create_cylinder(
        "Y", position, "$sma_pin_radius", "$sma_pin_length", name=f"sma_pin_{i+1}"
    )
    for i, position in enumerate(face_positions)
]
hfss.modeler.subtract("cavity", ["sma_pin_1", "sma_pin_2"], keep_originals=False)

# assign 50 ohm impedance to sma ports
faces = [
    hfss.modeler.get_faceid_from_position(
        ["$sma_pin_radius", "$cavity_y / 2 + $sma_port_length", "-$cavity_z / 4"]
    ),
    hfss.modeler.get_faceid_from_position(
        ["$sma_pin_radius", "$cavity_y / 2 + $sma_port_length", "$cavity_z / 4"]
    ),
]
omodule = hfss.odesign.GetModule("BoundarySetup")
omodule.AssignImpedance(
    [
        "NAME:sma",
        "Faces:=",
        faces,
        "Resistance:=",
        "50",
        "Reactance:=",
        "0",
        "InfGroundPlane:=",
        False,
    ]
)

# assign cryo copper conductivity to cavity
omodule.AssignFiniteCond(
    [
        "NAME:cryo_copper",
        "Faces:=",
        [face.id for face in cavity.faces if face.id not in faces],
        "UseMaterial:=",
        True,
        "Material:=",
        "cryo_copper",
        "UseThickness:=",
        False,
        "Roughness:=",
        "0um",
        "InfGroundPlane:=",
        False,
        "IsTwoSided:=",
        False,
        "IsInternal:=",
        True,
    ]
)

# rotate and fit all the contents in the view
hfss.modeler.rotate("cavity", 90)
hfss.modeler.fit_all()

In [ ]:
# run eigenmode analysis
if not "EM_Cavity" in hfss.get_setups():
    setup = {
        "MinimumFrequency": "2GHz",
        "NumModes": 10,
        "MaxDeltaFreq": 0.05,
        "ConvergeOnRealFreq": True,
        "MaximumPasses": 20,
        "MinimumPasses": 2,
        "MinimumConvergedPasses": 2,
        "BasisOrder": -1,
    }
    hfss.create_setup(name="EM_Cavity", **setup)

hfss.analyze(cores=num_cores)

In [ ]:
eigenmodes = hfss.post.available_report_quantities()
eigen_q = hfss.post.available_report_quantities(quantities_category="Eigen Q")
soln_data = {
    "Eigenmode": eigenmodes,
    "Frequency_Real (GHz)": [
        hfss.post.get_solution_data(expressions=mode).get_expression_data(formula="real")[1][0] / giga
        for mode in eigenmodes
    ],
    "Frequency_Imag (GHz)": [
        hfss.post.get_solution_data(expressions=mode).get_expression_data(formula="imag")[1][0] / giga
        for mode in eigenmodes
    ],
    "Q": [hfss.post.get_solution_data(expressions=q).get_expression_data(formula="real")[1][0] for q in eigen_q],
}
eigenmode_results = pd.DataFrame(soln_data)
eigenmode_results.to_csv("data/eigenmode/cavity_eigenmode_results.csv", index=False)
eigenmode_results

In [ ]:
def f_mnl(a, b, d, mode):
    m, n, l = mode
    return c * np.sqrt((m / a) ** 2 + (n / b) ** 2 + (l / d) ** 2) / 2


def f_n(L, n):
    return (2 * n + 1) * c / (4 * L)


# find a and d that realize mode 1 frequency of 6.66 GHz
b = parameters["cavity_y"] * milli
x = np.linspace(1, 100, int(1e6)) * milli
idx = np.argmin(np.abs(f_mnl(x, b, x, [1, 0, 1]) - 6.66 * giga))
a = d = parameters["cavity_x"] * milli

eigenmodes, f_sim, _, _ = (
    pd.read_csv("data/eigenmode/cavity_eigenmode_results.csv").to_numpy().T
)

fm = []
TE_cutoff = f_mnl(a, b, d, [1, 0, 1])
fm.append((TE_cutoff, [1, 0, 1]))

for m in range(0, 10):
    for n in range(0, 10):
        for l in range(0, 10):
            mode = [m, n, l]
            frequency = f_mnl(a, b, d, mode)
            if mode != [0, 0, l] and mode != [m, 0, 0]:
                if frequency > TE_cutoff:
                    fm.append((frequency, mode))

modes = np.array([sorted(fm)[i][1] for i in range(20)])[: len(f_sim)]
f_anal = np.array([sorted(fm)[i][0] for i in range(20)])[: len(f_sim)]

# append sma port modes
# there are two ports with the same pin lengths
sma_modes = np.array([f_n(sma_pin_length * 1e-3, 0), f_n(sma_pin_length * 1e-3, 0)])
f_anal = np.append(f_anal, sma_modes)
f_anal = np.sort(f_anal)[:-2]

fig, ax = plt.subplots()

ax.plot(f_anal / giga, "bs--", label=r"Analytical")
ax.plot(f_sim, "r^", label="Simulation")

ax.set_xticks(range(len(eigenmodes)))
ax.set_xticklabels(eigenmodes)
ax.set_ylabel(r"$\text{Frequency} \; (\mathrm{GHz})$")
ax.legend()
plt.savefig("figures/cavity_modes")
plt.show()

In [ ]:
# rectangular cavity TE modes
modes

## Snail

In [ ]:
chip_name = "snail"

if "EM_Snail" not in hfss.design_list:
    hfss.insert_design("EM_Snail", "Eigenmode")
else:
    hfss.set_active_design("EM_Snail")

if hfss.modeler.get_objects_in_group(chip_name):
    hfss.modeler.delete(chip_name)

# import and position snail chip
hfss.oeditor.Import(
    [
        "NAME:NativeBodyParameters",
        "HealOption:=",
        1,
        "Options:=",
        "-1",
        "FileType:=",
        "UnRecognized",
        "MaxStitchTol:=",
        -1,
        "ImportFreeSurfaces:=",
        False,
        "GroupByAssembly:=",
        False,
        "CreateGroup:=",
        True,
        "STLFileUnit:=",
        "Auto",
        "MergeFacesAngle:=",
        0.02,
        "HealSTL:=",
        False,
        "ReduceSTL:=",
        False,
        "ReduceMaxError:=",
        0,
        "ReducePercentage:=",
        100,
        "PointCoincidenceTol:=",
        1e-06,
        "CreateLightweightPart:=",
        False,
        "ImportMaterialNames:=",
        False,
        "SeparateDisjointLumps:=",
        False,
        "SourceFile:=",
        os.getcwd() + f"/hfss_files/snail/{chip_name}.sab",
    ]
)

hfss.modeler.move(
    chip_name,
    [
        -parameters["chip_delta_y_pos"],
        -parameters["chip_delta_x_neg"] - parameters["chip_padding"],
        -parameters["chip_thickness"],
    ],
)

hfss.modeler.rotate(chip_name, "Z", parameters["theta"] - 90)

# substrate material
hfss.oeditor.ChangeProperty(
    [
        "NAME:AllTabs",
        [
            "NAME:Geometry3DAttributeTab",
            ["NAME:PropServers", "substrate"],
            [
                "NAME:ChangedProps",
                ["NAME:Material", "Value:=", '"silicon - tensor"'],
                ["NAME:Transparent", "Value:=", 0.9],
            ],
        ],
    ]
)

# pads and ft boundaries
hfss.assign_perfecte_to_sheets(["pad_1", "pad_2"], "pads")
# hfss.assign_perfecte_to_sheets(["ft", "ft_coupler"], "ft")
hfss.assign_perfecte_to_sheets("ft", "ft")

# JJ boundary
jj_length = 30  # um
theta = parameters["theta"] * pi / 180
x0 = 0.5 * jj_length * np.cos(theta)
y0 = 0.5 * jj_length * np.sin(theta)
z0 = 0
start = [-x0 * milli, -y0 * milli, z0]
end = [x0 * milli, y0 * milli, z0]
L_J = 5.9e-9

hfss.assign_lumped_rlc_to_sheet(
    assignment="jj", start_direction=[start, end], name="jj", inductance=L_J
)

# fit all the contents in the view
hfss.modeler.fit_all()

In [ ]:
# mesh
hfss.mesh.assign_length_mesh(
    assignment="substrate",
    inside_selection=False,
    maximum_length="2mm",
    maximum_elements=None,
    name="substrate",
)
hfss.mesh.assign_length_mesh(
    assignment=["pad_1", "pad_2"],
    inside_selection=False,
    maximum_length="100um",
    maximum_elements=None,
    name="pads",
)
hfss.mesh.assign_length_mesh(
    assignment="jj",
    inside_selection=False,
    maximum_length="10um",
    maximum_elements=None,
    name="jj",
)
hfss.mesh.assign_length_mesh(
    assignment="ft",
    inside_selection=False,
    maximum_length="20um",
    maximum_elements=None,
    name="ft",
)
# hfss.mesh.assign_length_mesh(
#     assignment="ft_coupler",
#     inside_selection=False,
#     maximum_length="2um",
#     maximum_elements=None,
#     name="ft_coupler",
# )

In [ ]:
# additional meshing needed for ft
omodule = hfss.odesign.GetModule("MeshSetup")
omodule.AssignApplyCurvlinearElementsOp(
    ["NAME:ft_curvilinear", "Objects:=", ["ft"], "Apply:=", True]
)
omodule.AssignTrueSurfOp(
    [
        "NAME:ft_surf_approx",
        "Objects:=",
        ["ft"],
        "CurvedSurfaceApproxChoice:=",
        "UseSlider",
        "SliderMeshSettings:=",
        9,
    ]
)
omodule.AssignSurfPriorityForTauOp(
    ["NAME:ft_suface_priority", "Objects:=", ["ft"], "SurfaceRepPriority:=", 1]
)

In [ ]:
# create vaccum box
if "vacuum_box" in hfss.modeler.object_names:
    hfss.modeler.delete("vacuum_box")

box_origin = [0, 0, "-2.5mm"]
box_dim = ["$chip_length", "$chip_width", "$chip_thickness + 5mm"]
vacuum_box_params = {
    "origin": box_origin,
    "sizes": box_dim,
    "name": "vacuum_box",
    "material": "vacuum",
}
vacuum_box = hfss.modeler.create_box(**vacuum_box_params)
vacuum_box.move(
    [
        -parameters["chip_delta_y_pos"],
        -parameters["chip_delta_x_neg"] - parameters["chip_padding"],
        -parameters["chip_thickness"],
    ]
)
vacuum_box.rotate("Z", parameters["theta"] - 90)

# fit all the contents in the view
hfss.modeler.fit_all()

In [ ]:
# run eigenmode analysis
if not "EM_Snail" in hfss.get_setups():
    setup = {
        "MinimumFrequency": "2GHz",
        "NumModes": 5,
        "MaxDeltaFreq": 0.1,
        "ConvergeOnRealFreq": True,
        "MaximumPasses": 20,
        "MinimumPasses": 2,
        "MinimumConvergedPasses": 2,
        "BasisOrder": -1,
    }
    hfss.create_setup(name="EM_Snail", **setup)

hfss.analyze(cores=num_cores)

In [ ]:
eigenmodes = hfss.post.available_report_quantities()
eigen_q = hfss.post.available_report_quantities(quantities_category="Eigen Q")
soln_data = {
    "Eigenmode": eigenmodes,
    "Frequency_Real (GHz)": [
        hfss.post.get_solution_data(expressions=mode).get_expression_data(formula="real")[1][0] / giga
        for mode in eigenmodes
    ],
    "Frequency_Imag (GHz)": [
        hfss.post.get_solution_data(expressions=mode).get_expression_data(formula="imag")[1][0] / giga
        for mode in eigenmodes
    ],
    "Q": [hfss.post.get_solution_data(expressions=q).get_expression_data(formula="real")[1][0] for q in eigen_q],
}
eigenmode_results = pd.DataFrame(soln_data)
eigenmode_results.to_csv("data/eigenmode/snail_eigenmode_results.csv", index=False)
eigenmode_results

## Cavity with Snail 

In [ ]:
# set design
if "EM_Cavity_with_Snail" not in hfss.design_list:
    hfss.insert_design("EM_Cavity_with_Snail", "Eigenmode")
else:
    hfss.set_active_design("EM_Cavity_with_Snail")

# delete all objects
if hfss.modeler.get_objects_in_group("snail"):
    hfss.modeler.delete("snail")

if hfss.modeler.object_list:
    hfss.modeler.delete()

# import step file
hfss.modeler.import_3d_cad("inventor_files/snail_coffin_cavity.stp")

# create bounding box
box = hfss.modeler.get_group_bounding_box("snail_coffin_cavity")
box_min = box[:3]
box_max = box[3:]
box_origin = box_min
box_dim = [a - b for a, b in zip(box_max, box_min)]
cavity_params = {
    "origin": box_origin,
    "sizes": box_dim,
    "name": "cavity",
    "material": "vacuum",
}
cavity = hfss.modeler.create_box(**cavity_params)

# subtract
objs = hfss.modeler.object_names
hfss.modeler.subtract(objs[2], [objs[0], objs[1]], keep_originals=False)
hfss.modeler.ungroup("snail_coffin_cavity")

# sma pins
face_positions = [
    [0, "$cavity_y/2 + $sma_port_length - $sma_pin_length", "-$cavity_z/4"],
    [0, "$cavity_y/2 + $sma_port_length - $sma_pin_length", "$cavity_z/4"],
]
[
    hfss.modeler.create_cylinder(
        "Y", position, "$sma_pin_radius", "$sma_pin_length", name=f"sma_pin_{i+1}"
    )
    for i, position in enumerate(face_positions)
]
hfss.modeler.subtract("cavity", ["sma_pin_1", "sma_pin_2"], keep_originals=False)

# assign 50 ohm impedance to sma ports
faces = [
    hfss.modeler.get_faceid_from_position(
        ["$sma_pin_radius", "$cavity_y / 2 + $sma_port_length", "-$cavity_z / 4"]
    ),
    hfss.modeler.get_faceid_from_position(
        ["$sma_pin_radius", "$cavity_y / 2 + $sma_port_length", "$cavity_z / 4"]
    ),
]
omodule = hfss.odesign.GetModule("BoundarySetup")
omodule.AssignImpedance(
    [
        "NAME:sma",
        "Faces:=",
        faces,
        "Resistance:=",
        "50",
        "Reactance:=",
        "0",
        "InfGroundPlane:=",
        False,
    ]
)

# assign cryo copper conductivity to cavity
omodule.AssignFiniteCond(
    [
        "NAME:cryo_copper",
        "Faces:=",
        [face.id for face in cavity.faces if face.id not in faces],
        "UseMaterial:=",
        True,
        "Material:=",
        "cryo_copper",
        "UseThickness:=",
        False,
        "Roughness:=",
        "0um",
        "InfGroundPlane:=",
        False,
        "IsTwoSided:=",
        False,
        "IsInternal:=",
        True,
    ]
)

# rotate and fit all the contents in the view
hfss.modeler.rotate("cavity", 90)
hfss.modeler.fit_all()

In [ ]:
chip_name = "snail"

if hfss.modeler.get_objects_in_group(chip_name):
    hfss.modeler.delete(chip_name)

# import and position snail chip
hfss.oeditor.Import(
    [
        "NAME:NativeBodyParameters",
        "HealOption:=",
        1,
        "Options:=",
        "-1",
        "FileType:=",
        "UnRecognized",
        "MaxStitchTol:=",
        -1,
        "ImportFreeSurfaces:=",
        False,
        "GroupByAssembly:=",
        False,
        "CreateGroup:=",
        True,
        "STLFileUnit:=",
        "Auto",
        "MergeFacesAngle:=",
        0.02,
        "HealSTL:=",
        False,
        "ReduceSTL:=",
        False,
        "ReduceMaxError:=",
        0,
        "ReducePercentage:=",
        100,
        "PointCoincidenceTol:=",
        1e-06,
        "CreateLightweightPart:=",
        False,
        "ImportMaterialNames:=",
        False,
        "SeparateDisjointLumps:=",
        False,
        "SourceFile:=",
        os.getcwd() + f"/hfss_files/snail/{chip_name}.sab",
    ]
)

hfss.modeler.move(
    chip_name,
    [
        -parameters["chip_delta_y_pos"],
        -parameters["chip_delta_x_neg"] - parameters["chip_padding"],
        -parameters["chip_thickness"],
    ],
)

hfss.modeler.rotate(chip_name, "Z", parameters["theta"] - 90)

# substrate material
hfss.oeditor.ChangeProperty(
    [
        "NAME:AllTabs",
        [
            "NAME:Geometry3DAttributeTab",
            ["NAME:PropServers", "substrate"],
            [
                "NAME:ChangedProps",
                ["NAME:Material", "Value:=", '"silicon - tensor"'],
                ["NAME:Transparent", "Value:=", 0.9],
            ],
        ],
    ]
)

# pads and ft boundaries
hfss.assign_perfecte_to_sheets(["pad_1", "pad_2"], "pads")
# hfss.assign_perfecte_to_sheets(["ft", "ft_coupler"], "ft")
hfss.assign_perfecte_to_sheets("ft", "ft")

# JJ boundary
jj_length = 30  # um
theta = parameters["theta"] * pi / 180
x0 = 0.5 * jj_length * np.cos(theta)
y0 = 0.5 * jj_length * np.sin(theta)
z0 = 0
start = [-x0 * milli, -y0 * milli, z0]
end = [x0 * milli, y0 * milli, z0]
L_J = 5.9e-9

hfss.assign_lumped_rlc_to_sheet(
    assignment="jj", start_direction=[start, end], name="jj", inductance=L_J
)

# fit all the contents in the view
hfss.modeler.fit_all()

In [ ]:
# mesh
hfss.mesh.assign_length_mesh(
    assignment="substrate",
    inside_selection=False,
    maximum_length="2mm",
    maximum_elements=None,
    name="substrate",
)
hfss.mesh.assign_length_mesh(
    assignment=["pad_1", "pad_2"],
    inside_selection=False,
    maximum_length="100um",
    maximum_elements=None,
    name="pads",
)
hfss.mesh.assign_length_mesh(
    assignment="jj",
    inside_selection=False,
    maximum_length="10um",
    maximum_elements=None,
    name="jj",
)
hfss.mesh.assign_length_mesh(
    assignment="ft",
    inside_selection=False,
    maximum_length="20um",
    maximum_elements=None,
    name="ft",
)
# hfss.mesh.assign_length_mesh(
#     assignment="ft_coupler",
#     inside_selection=False,
#     maximum_length="3um",
#     maximum_elements=None,
#     name="ft_coupler",
# )

In [ ]:
# additional meshing needed for ft
omodule = hfss.odesign.GetModule("MeshSetup")
omodule.AssignApplyCurvlinearElementsOp(
    ["NAME:ft_curvilinear", "Objects:=", ["ft"], "Apply:=", True]
)
omodule.AssignTrueSurfOp(
    [
        "NAME:ft_surf_approx",
        "Objects:=",
        ["ft"],
        "CurvedSurfaceApproxChoice:=",
        "UseSlider",
        "SliderMeshSettings:=",
        9,
    ]
)
omodule.AssignSurfPriorityForTauOp(
    ["NAME:ft_suface_priority", "Objects:=", ["ft"], "SurfaceRepPriority:=", 1]
)

In [ ]:
# run eigenmode analysis
if not "EM_Cavity_with_Snail" in hfss.get_setups():
    setup = {
        "MinimumFrequency": "2GHz",
        "NumModes": 10,
        "MaxDeltaFreq": 0.1,
        "ConvergeOnRealFreq": True,
        "MaximumPasses": 10,
        "MinimumPasses": 2,
        "MinimumConvergedPasses": 2,
        "BasisOrder": -1,
    }
    hfss.create_setup(name="EM_Cavity_with_Snail", **setup)

hfss.analyze(cores=num_cores)

In [ ]:
eigenmodes = hfss.post.available_report_quantities()
eigen_q = hfss.post.available_report_quantities(quantities_category="Eigen Q")
soln_data = {
    "Eigenmode": eigenmodes,
    "Frequency_Real (GHz)": [
        hfss.post.get_solution_data(expressions=mode).get_expression_data(formula="real")[1][0] / giga
        for mode in eigenmodes
    ],
    "Frequency_Imag (GHz)": [
        hfss.post.get_solution_data(expressions=mode).get_expression_data(formula="imag")[1][0] / giga
        for mode in eigenmodes
    ],
    "Q": [hfss.post.get_solution_data(expressions=q).get_expression_data(formula="real")[1][0] for q in eigen_q],
}
eigenmode_results = pd.DataFrame(soln_data)
eigenmode_results.to_csv(
    "data/eigenmode/cavity_and_snail_eigenmode_results.csv", index=False
)
eigenmode_results

# Driven Modal

In [ ]:
# set design
if "DM" not in hfss.design_list:
    hfss.insert_design("DM", "Modal")
    hfss.hybrid = True
else:
    hfss.set_active_design("DM")

# delete all objects
if hfss.modeler.get_objects_in_group("snail"):
    hfss.modeler.delete("snail")

if hfss.modeler.object_list:
    hfss.modeler.delete()

# import step file
hfss.modeler.import_3d_cad("inventor_files/snail_coffin_cavity.stp")

# create bounding box
box = hfss.modeler.get_group_bounding_box("snail_coffin_cavity")
box_min = box[:3]
box_max = box[3:]
box_origin = box_min
box_dim = [a - b for a, b in zip(box_max, box_min)]
cavity_params = {
    "origin": box_origin,
    "sizes": box_dim,
    "name": "cavity",
    "material": "vacuum",
}
cavity = hfss.modeler.create_box(**cavity_params)

# subtract
objs = hfss.modeler.object_names
hfss.modeler.subtract(objs[2], [objs[0], objs[1]], keep_originals=False)
hfss.modeler.ungroup("snail_coffin_cavity")

# sma pins
face_positions = [
    [0, "$cavity_y/2 + $sma_port_length - $sma_pin_length", "-$cavity_z/4"],
    [0, "$cavity_y/2 + $sma_port_length - $sma_pin_length", "$cavity_z/4"],
]
[
    hfss.modeler.create_cylinder(
        "Y", position, "$sma_pin_radius", "$sma_pin_length", name=f"sma_pin_{i+1}"
    )
    for i, position in enumerate(face_positions)
]
hfss.modeler.subtract("cavity", ["sma_pin_1", "sma_pin_2"], keep_originals=False)

# assign 50 ohm impedance to sma ports
faces = [
    hfss.modeler.get_faceid_from_position(
        ["$sma_pin_radius", "$cavity_y / 2 + $sma_port_length", "-$cavity_z / 4"]
    ),
    hfss.modeler.get_faceid_from_position(
        ["$sma_pin_radius", "$cavity_y / 2 + $sma_port_length", "$cavity_z / 4"]
    ),
]
omodule = hfss.odesign.GetModule("BoundarySetup")
omodule.AssignImpedance(
    [
        "NAME:sma",
        "Faces:=",
        faces,
        "Resistance:=",
        "50",
        "Reactance:=",
        "0",
        "InfGroundPlane:=",
        False,
    ]
)

# assign cryo copper conductivity to cavity
omodule.AssignFiniteCond(
    [
        "NAME:cryo_copper",
        "Faces:=",
        [face.id for face in cavity.faces if face.id not in faces],
        "UseMaterial:=",
        True,
        "Material:=",
        "cryo_copper",
        "UseThickness:=",
        False,
        "Roughness:=",
        "0um",
        "InfGroundPlane:=",
        False,
        "IsTwoSided:=",
        False,
        "IsInternal:=",
        True,
    ]
)

# rotate and fit all the contents in the view
hfss.modeler.rotate("cavity", 90)
hfss.modeler.fit_all()

In [ ]:
chip_name = "snail"

if hfss.modeler.get_objects_in_group(chip_name):
    hfss.modeler.delete(chip_name)

# import and position snail chip
hfss.oeditor.Import(
    [
        "NAME:NativeBodyParameters",
        "HealOption:=",
        1,
        "Options:=",
        "-1",
        "FileType:=",
        "UnRecognized",
        "MaxStitchTol:=",
        -1,
        "ImportFreeSurfaces:=",
        False,
        "GroupByAssembly:=",
        False,
        "CreateGroup:=",
        True,
        "STLFileUnit:=",
        "Auto",
        "MergeFacesAngle:=",
        0.02,
        "HealSTL:=",
        False,
        "ReduceSTL:=",
        False,
        "ReduceMaxError:=",
        0,
        "ReducePercentage:=",
        100,
        "PointCoincidenceTol:=",
        1e-06,
        "CreateLightweightPart:=",
        False,
        "ImportMaterialNames:=",
        False,
        "SeparateDisjointLumps:=",
        False,
        "SourceFile:=",
        os.getcwd() + f"/hfss_files/snail/{chip_name}.sab",
    ]
)

hfss.modeler.move(
    chip_name,
    [
        -parameters["chip_delta_y_pos"],
        -parameters["chip_delta_x_neg"] - parameters["chip_padding"],
        -parameters["chip_thickness"],
    ],
)

hfss.modeler.rotate(chip_name, "Z", parameters["theta"] - 90)

# substrate material
hfss.oeditor.ChangeProperty(
    [
        "NAME:AllTabs",
        [
            "NAME:Geometry3DAttributeTab",
            ["NAME:PropServers", "substrate"],
            [
                "NAME:ChangedProps",
                ["NAME:Material", "Value:=", '"silicon - tensor"'],
                ["NAME:Transparent", "Value:=", 0.9],
            ],
        ],
    ]
)

# pads and ft boundaries
hfss.assign_perfecte_to_sheets(["pad_1", "pad_2"], "pads")
# hfss.assign_perfecte_to_sheets(["ft", "ft_coupler"], "ft")
hfss.assign_perfecte_to_sheets("ft", "ft")

# JJ excitation
jj_length = 30  # um
theta = parameters["theta"] * pi / 180
x0 = 0.5 * jj_length * np.cos(theta)
y0 = 0.5 * jj_length * np.sin(theta)
z0 = 0
start = [-x0 * milli, -y0 * milli, z0]
end = [x0 * milli, y0 * milli, z0]
Z0 = 1e6

hfss._create_lumped_driven(
    assignment="jj",
    int_line_start=start,
    int_line_stop=end,
    impedance=Z0,
    port_name="jj",
    renorm=False,
    deemb=False,
)

# fit all the contents in the view
hfss.modeler.fit_all()

In [ ]:
# mesh
hfss.mesh.assign_length_mesh(
    assignment="substrate",
    inside_selection=False,
    maximum_length="2mm",
    maximum_elements=None,
    name="substrate",
)
hfss.mesh.assign_length_mesh(
    assignment=["pad_1", "pad_2"],
    inside_selection=False,
    maximum_length="100um",
    maximum_elements=None,
    name="pads",
)
hfss.mesh.assign_length_mesh(
    assignment="jj",
    inside_selection=False,
    maximum_length="10um",
    maximum_elements=None,
    name="jj",
)
hfss.mesh.assign_length_mesh(
    assignment="ft",
    inside_selection=False,
    maximum_length="20um",
    maximum_elements=None,
    name="ft",
)
# hfss.mesh.assign_length_mesh(
#     assignment="ft_coupler",
#     inside_selection=False,
#     maximum_length="2um",
#     maximum_elements=None,
#     name="ft_coupler",
# )

In [ ]:
# additional meshing needed for ft
omodule = hfss.odesign.GetModule("MeshSetup")
omodule.AssignApplyCurvlinearElementsOp(
    ["NAME:ft_curvilinear", "Objects:=", ["ft"], "Apply:=", True]
)
omodule.AssignTrueSurfOp(
    [
        "NAME:ft_surf_approx",
        "Objects:=",
        ["ft"],
        "CurvedSurfaceApproxChoice:=",
        "UseSlider",
        "SliderMeshSettings:=",
        9,
    ]
)
omodule.AssignSurfPriorityForTauOp(
    ["NAME:ft_suface_priority", "Objects:=", ["ft"], "SurfaceRepPriority:=", 1]
)

In [ ]:
# run driven modal analysis
if "DM" in hfss.get_setups():
    hfss.delete_setup("DM")

setup = {
    "SolveType": "Single",
    "Frequency": "7GHz",
    "MaxDeltaE": 0.01,
    "MaximumPasses": 10,
    "MinimumPasses": 3,
    "MinimumConvergedPasses": 3,
    "PercentRefinement": 30,
    "IsEnabled": True,
    "BasisOrder": -1,
    "DoLambdaRefine": True,
    "DoMaterialLambda": True,
    "SetLambdaTarget": False,
    "Target": 0.3333,
    "UseMaxTetIncrease": False,
    "DrivenSolverType": "Direct Solver",
    "EnhancedLowFreqAccuracy": False,
    "SaveRadFieldsOnly": False,
    "SaveAnyFields": True,
    "IESolverType": "Auto",
    "LambdaTargetForIESolver": 0.15,
    "UseDefaultLambdaTgtForIESolver": True,
    "IE Solver Accuracy": "Balanced",
    "InfiniteSphereSetup": "",
}
hfss.create_setup(name="DM", **setup)

for i, freq in enumerate(np.arange(0.1, 10, 0.05)):
    if not f"Sweep{i}" in hfss.get_sweeps("DM"):
        hfss.create_linear_count_sweep(
            setup="DM",
            units="GHz",
            start_frequency=freq,
            stop_frequency=freq + 0.05,
            num_of_freq_points=25000,
            name=f"Sweep_{i}",
            save_fields=False,
            sweep_type="Fast",
        )

hfss.analyze(cores=num_cores)

In [ ]:
# get data
freq = []
re_Y_c = []
im_Y_c = []

sweeps = hfss.get_sweeps("DM")

for sweep in sweeps:
    data = hfss.post.get_solution_data("Y(jj,jj)", setup_sweep_name=sweep)
    freq.append(data.primary_sweep_values)
    re_Y_c.append(data.get_expression_data("Y(jj,jj)", formula="real")[1])
    im_Y_c.append(data.get_expression_data("Y(jj,jj)", formula="imag")[1])

In [ ]:
freq = np.array(freq).flatten()
im_Y_c = np.array(im_Y_c).flatten()
re_Y_c = np.array(re_Y_c).flatten()
Y_c = re_Y_c + 1j * im_Y_c

In [ ]:
# save data
pd.DataFrame(
    {
        "Frequency (GHz)": freq,
        "Re(Y_c) (S)": re_Y_c,
        "Im(Y_c) (S)": im_Y_c,
        "Y_c (S)": Y_c,
    }
).to_csv(f"data/driven_modal/admittances_{parameters['theta']:.0f}deg.csv", index=False)

In [ ]:
df = pd.read_csv("data/driven_modal/admittances_5deg.csv")
freq = df["Frequency (GHz)"]
im_Y_c = df["Im(Y_c) (S)"]

In [ ]:
fig, ax = plt.subplots()

ax.plot(freq, im_Y_c, "b", label=r"$\mathrm{Im}{Y_\text{c}}$")

ax.set_xlabel(r"$\text{Frequency} \; (\mathrm{GHz})$")
ax.set_ylabel(r"$\text{Admittance} \; (\mathrm{S})$")

ax.set_xlim(4, 10)
ax.set_ylim(-0.1, 0.1)

ax.legend()
plt.savefig("figures/im_Y_c_5deg")
plt.show()

# Save & Release HFSS

In [ ]:
hfss.save_project()

In [ ]:
hfss.release_desktop()

# Black Box Quantization

In [ ]:
L_J = 5.73e-9
C_J = 4e-15

theta = 5
df = pd.read_csv(f"data/driven_modal/admittances_{theta:.0f}deg.csv")
freq = df["Frequency (GHz)"].to_numpy() * 1e9
Y_c = df["Y_c (S)"].to_numpy(dtype=complex)

In [ ]:
bbq = BBQAnalysis(Y_c, freq, L_J, C_J)
lcrzphi = bbq.calculate_LCRZPhi()
bbq_info = bbq.save_BBOX_info(filename=f"data/bbq/bbq_results_{theta:.0f}deg.csv")
bbq_info

In [ ]:
fig, ax = plt.subplots()

bbq.plot_admittances(fig=fig, ax=ax, ylim=[-0.2, 0.2])

ax.set_xlim([3, 10])

plt.savefig(f"figures/admittances_{theta:.0f}deg")
plt.show()

In [ ]:
fig, ax = plt.subplots()

bbq.plot_gs_vs_freq(fig=fig, ax=ax)

plt.savefig(f"figures/gs_{theta:.0f}deg")
plt.show()

# Theta Sweep

In [40]:
# This for loop suuuuuuuuuuuuuuuuucccckkkks

# set design


if "DM" not in hfss.design_list:


    hfss.insert_design("DM", "Modal")


    hfss.hybrid = True


else:


    hfss.set_active_design("DM")


name = "theta"



initial_value = parameters[name]



values = [40, 60]



for value in values:


    # Connect to Inventor


    inv = win32com.client.Dispatch("Inventor.Application")


    # Open the part file


    doc = inv.Documents.Open(
        os.getcwd() + "\\inventor_files\\snail_coffin_cavity_bottom.ipt"
    )


    # Access the parameters collection


    params = doc.ComponentDefinition.Parameters.UserParameters


    # Find the design variable by name


    design_variable = params.Item(name)


    # Change the value of the design variable


    design_variable.Value = value * pi / 180


    # Save the changes to the part file


    doc.Update()


    doc.Save()


    # Close the part document


    doc.Close()


    doc = inv.Documents.Open(
        os.getcwd() + "\\inventor_files\\snail_coffin_cavity_top.ipt"
    )


    doc.Update()


    doc.Save()


    doc.Close()


    # Open the assembly file


    doc = inv.Documents.Open(os.getcwd() + "\\inventor_files\\snail_coffin_cavity.iam")


    # Update the assembly to reflect the changes in the part


    doc.Update()


    doc.Save()


    # Save the assembly as a STEP file


    step_file_path = os.getcwd() + "\\inventor_files\\snail_coffin_cavity.stp"


    doc.SaveAs(step_file_path, 11)  # 11 corresponds to STEP format


    # Close the assembly document


    doc.Close()


    inv.Quit()


    # Release the COM objects


    doc = None


    params = None


    inv = None


    # Force garbage collection


    gc.collect()


    # hfss


    if "DM" in hfss.get_setups():


        hfss.delete_setup("DM")


    hfss["$theta"] = f"{value}deg"


    # cavity


    # delete all objects


    if hfss.modeler.get_objects_in_group("snail"):


        hfss.modeler.delete("snail")


    if hfss.modeler.object_list:


        hfss.modeler.delete()


    # import step file


    hfss.modeler.import_3d_cad("inventor_files/snail_coffin_cavity.stp")


    # create bounding box


    box = hfss.modeler.get_group_bounding_box("snail_coffin_cavity")


    box_min = box[:3]


    box_max = box[3:]


    box_origin = box_min


    box_dim = [a - b for a, b in zip(box_max, box_min)]


    cavity_params = {
        "origin": box_min,
        "sizes": box_dim,
        "name": "cavity",
        "material": "vacuum",
    }


    cavity = hfss.modeler.create_box(**cavity_params)


    # subtract


    objs = hfss.modeler.object_names


    hfss.modeler.subtract(objs[2], [objs[0], objs[1]], keep_originals=False)


    hfss.modeler.ungroup("snail_coffin_cavity")


    # sma pins


    face_positions = [
        [0, "$cavity_y/2 + $sma_port_length - $sma_pin_length", "-$cavity_z/4"],
        [0, "$cavity_y/2 + $sma_port_length - $sma_pin_length", "$cavity_z/4"],
    ]


    [
        hfss.modeler.create_cylinder(
            "Y", position, "$sma_pin_radius", "$sma_pin_length", name=f"sma_pin_{i+1}"
        )
        for i, position in enumerate(face_positions)
    ]


    hfss.modeler.subtract("cavity", ["sma_pin_1", "sma_pin_2"], keep_originals=False)


    # assign 50 ohm impedance to sma ports


    faces = [
        hfss.modeler.get_faceid_from_position(
            ["$sma_pin_radius", "$cavity_y / 2 + $sma_port_length", "-$cavity_z / 4"]
        ),
        hfss.modeler.get_faceid_from_position(
            ["$sma_pin_radius", "$cavity_y / 2 + $sma_port_length", "$cavity_z / 4"]
        ),
    ]


    omodule = hfss.odesign.GetModule("BoundarySetup")


    omodule.AssignImpedance(
        [
            "NAME:sma",
            "Faces:=",
            faces,

            "Resistance:=",
            "50",
            "Reactance:=",
            "0",
            "InfGroundPlane:=",
            False,
        ]
    )


    # assign cryo copper conductivity to cavity


    omodule.AssignFiniteCond(
        [
            "NAME:cryo_copper",
            "Faces:=",
            [face.id for face in cavity.faces if face.id not in faces],
            "UseMaterial:=",
            True,
            "Material:=",
            "cryo_copper",
            "UseThickness:=",
            False,
            "Roughness:=",
            "0um",
            "InfGroundPlane:=",
            False,
            "IsTwoSided:=",
            False,
            "IsInternal:=",
            True,
        ]
    )


    # rotate and fit all the contents in the view


    hfss.modeler.rotate("cavity", 90)


    hfss.modeler.fit_all()


    # Snail


    if hfss.modeler.get_objects_in_group("snail"):


        hfss.modeler.delete("snail")


    # import and position snail chip


    hfss.oeditor.Import(
        [
            "NAME:NativeBodyParameters",
            "HealOption:=",
            1,
            "Options:=",
            "-1",
            "FileType:=",
            "UnRecognized",
            "MaxStitchTol:=",
            -1,
            "ImportFreeSurfaces:=",
            False,
            "GroupByAssembly:=",
            False,
            "CreateGroup:=",
            True,
            "STLFileUnit:=",
            "Auto",
            "MergeFacesAngle:=",
            0.02,
            "HealSTL:=",
            False,
            "ReduceSTL:=",
            False,
            "ReduceMaxError:=",
            0,
            "ReducePercentage:=",
            100,
            "PointCoincidenceTol:=",
            1e-06,
            "CreateLightweightPart:=",
            False,
            "ImportMaterialNames:=",
            False,
            "SeparateDisjointLumps:=",
            False,
            "SourceFile:=",
            os.getcwd() + "/hfss_files/snail/snail.sab",
        ]
    )


    hfss.modeler.move(
        "snail",

        [
            -parameters["chip_delta_y_pos"],
            -parameters["chip_delta_x_neg"] - parameters["chip_padding"],
            -parameters["chip_thickness"],
        ],
    )


    hfss.modeler.rotate("snail", "Z", value - 90)


    # substrate material


    hfss.oeditor.ChangeProperty(
        [
            "NAME:AllTabs",
            [
                "NAME:Geometry3DAttributeTab",
                ["NAME:PropServers", "substrate"],
                [
                    "NAME:ChangedProps",
                    ["NAME:Material", "Value:=", '"silicon - tensor"'],
                    ["NAME:Transparent", "Value:=", 0.9],
                ],
            ],
        ]
    )


    # pads and ft boundaries


    hfss.assign_perfecte_to_sheets(["pad_1", "pad_2"], "pads")


    # hfss.assign_perfecte_to_sheets(["ft", "ft_coupler"], "ft")


    hfss.assign_perfecte_to_sheets("ft", "ft")


    # JJ excitation


    jj_length = 30  # um


    theta = value * pi / 180


    x0 = 0.5 * jj_length * np.cos(theta)


    y0 = 0.5 * jj_length * np.sin(theta)


    z0 = 0


    start = [-x0 * milli, -y0 * milli, z0]


    end = [x0 * milli, y0 * milli, z0]


    Z0 = 1e6


    hfss._create_lumped_driven(
        assignment="jj",
        int_line_start=start,
        int_line_stop=end,

        impedance=Z0,
        port_name="jj",
        renorm=False,
        deemb=False,
    )


    # ft vacuum box


    # if "ft_vac_box" in hfss.modeler.object_names:


    #     hfss.modeler.delete("ft_vac_box")


    # ft_vac_box = hfss.modeler.create_box(


    #     ["15um", "-50um", 0], ["10um", "100um", "2um"], "ft_vac_box", "vacuum"


    # )


    # ft_vac_box.rotate("Z", parameters["theta"] - 90)


    # fit all the contents in the view


    hfss.modeler.fit_all()


    # mesh


    hfss.mesh.assign_length_mesh(
        assignment="substrate",
        inside_selection=False,
        maximum_length="2mm",
        maximum_elements=None,
        name="substrate",
    )


    hfss.mesh.assign_length_mesh(
        assignment=["pad_1", "pad_2"],
        inside_selection=False,
        maximum_length="100um",
        maximum_elements=None,
        name="pads",

    )


    hfss.mesh.assign_length_mesh(
        assignment="jj",
        inside_selection=False,
        maximum_length="10um",
        maximum_elements=None,
        name="jj",
    )


    hfss.mesh.assign_length_mesh(
        assignment="ft",
        inside_selection=False,
        maximum_length="20um",
        maximum_elements=None,
        name="ft",

    )


    # hfss.mesh.assign_length_mesh(


    #     assignment="ft_coupler",


    #     inside_selection=False,


    #     maximum_length="2um",


    #     maximum_elements=None,


    #     name="ft_coupler",


    # )


    # hfss.mesh.assign_length_mesh(


    #     names="ft_vac_box",


    #     isinside=False,


    #     maxlength="4um",


    #     maxel=None,


    #     meshop_name="ft_vac_box",


    # )


    # additional meshing needed for ft


    omodule = hfss.odesign.GetModule("MeshSetup")


    omodule.AssignApplyCurvlinearElementsOp(
        ["NAME:ft_curvilinear", "Objects:=", ["ft"], "Apply:=", True]
    )


    omodule.AssignTrueSurfOp(
        [
            "NAME:ft_surf_approx",
            "Objects:=",
            ["ft"],
            "CurvedSurfaceApproxChoice:=",
            "UseSlider",
            "SliderMeshSettings:=",
            9,
        ]
    )


    omodule.AssignSurfPriorityForTauOp(
        ["NAME:ft_suface_priority", "Objects:=", ["ft"], "SurfaceRepPriority:=", 1]
    )


    # run driven modal analysis


    if not "DM" in hfss.get_setups():


        setup = {
            "SolveType": "Single",
            "Frequency": "7GHz",
            "MaxDeltaE": 0.01,
            "MaximumPasses": 10,
            "MinimumPasses": 3,
            "MinimumConvergedPasses": 3,
            "PercentRefinement": 30,
            "IsEnabled": True,
            "BasisOrder": -1,
            "DoLambdaRefine": True,
            "DoMaterialLambda": True,
            "SetLambdaTarget": False,
            "Target": 0.3333,
            "UseMaxTetIncrease": False,
            "DrivenSolverType": "Direct Solver",
            "EnhancedLowFreqAccuracy": False,
            "SaveRadFieldsOnly": False,
            "SaveAnyFields": True,
            "IESolverType": "Auto",
            "LambdaTargetForIESolver": 0.15,
            "UseDefaultLambdaTgtForIESolver": True,
            "IE Solver Accuracy": "Balanced",
            "InfiniteSphereSetup": "",
        }


        hfss.create_setup(name="DM", **setup)


    for i, freq in enumerate(np.arange(0.1, 10, 0.05)):


        if not f"Sweep{i}" in hfss.get_sweeps("DM"):


            hfss.create_linear_count_sweep(
                setup="DM",
                units="GHz",
                start_frequency=freq,
                stop_frequency=freq + 0.05,
                num_of_freq_points=25000,
                name=f"Sweep_{i}",
                save_fields=False,
                sweep_type="Fast",
            )


    hfss.analyze(cores=num_cores)


    # data


    freq = []


    re_Y_c = []


    im_Y_c = []


    sweeps = hfss.get_sweeps("DM")


    for sweep in sweeps:


        data = hfss.post.get_solution_data("Y(jj,jj)", setup_sweep_name=sweep)


        freq.append(data.primary_sweep_values)


        re_Y_c.append(data.get_expression_data("Y(jj,jj)", formula="real")[1])


        im_Y_c.append(data.get_expression_data("Y(jj,jj)", formula="imag")[1])


    freq = np.array(freq).flatten()


    im_Y_c = np.array(im_Y_c).flatten()


    re_Y_c = np.array(re_Y_c).flatten()


    Y_c = re_Y_c + 1j * im_Y_c


    pd.DataFrame(
        {
            "Frequency (GHz)": freq,
            "Re(Y_c) (S)": re_Y_c,
            "Im(Y_c) (S)": im_Y_c,
            "Y_c (S)": Y_c,
        }
    ).to_csv(f"data/driven_modal/admittances_{value}deg.csv", index=False)